In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as mpl
from matplotlib import pyplot as plt
from scipy.spatial.distance import pdist, squareform

import warnings
import os
import pickle

from loader import load, get_formatted_data, get_raw_feature, DATA_DIR
from free_recall import create_fr_plots, split_by_feature, get_bounds
from fitness import format_text

In [3]:
#pip install scipy
#%pip install datawrangler

In [4]:
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 12

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [5]:
data = get_formatted_data()
_, fitbit, _ = load(recent=7, baseline=30)

# do some cleaning up to make parsing easier in this notebook
columns = [': '.join(c) if len(c[0]) > 0 else c[1] for c in fitbit.columns]
columns = [c.replace(' / ', '/') for c in columns]
fitbit = pd.DataFrame(index=fitbit.index, columns=columns, data=fitbit.values).drop(columns=[c for c in columns if 'efficiency' in c])
fitbit['sleep duration: recent'] /= (60 * 60 * 1000) # convert ms to hours

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/katemarine/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/katemarine/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/katemarine/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/katemarine/nltk_data...


Exception: throwing this error to help with debugging...

In [ ]:
with open(os.path.join(DATA_DIR, 'behavioral_summary.pkl'), 'rb') as f:
    behavioral = pickle.load(f)
bounds = [0, 25, 50, 75, 100]

In [ ]:
tasks = ['Free recall', 'Naturalistic recall', 'Foreign language flashcards', 'Spatial learning']
conditions = ['Immediate', 'Delayed']

bounds = [0, 25, 50, 75, 100]
task_colors = {t: c for t, c in zip(tasks, ['#3FA9F5', '#7AC943', '#FF931E', '#FF1D25'])}
condition_colors = ['#CCCCCC', '#1A1A1A']

task_cmaps = {t: sns.color_palette(f'light:{c}', n_colors=len(bounds)) for t, c in task_colors.items()}
task_cmaps_extended = {t: sns.color_palette(f'light:{c}', as_cmap=True) for t, c in task_colors.items()}

# Load in fitness data

Get summaries of recent measurements and recent vs. baseline measurements:
  - Activity (steps, zone minutes, floors/elevation)
  - Resting heart rate
  - Sleep

In [ ]:
fitness_tags = ['baseline' if 'baseline' in c else 'recent' if 'recent' in c else 'static' for c in fitbit.columns]
fitbit

In [ ]:
behavioral_tags = ['delayed' if 'delayed' in c else 'immediate' for c in behavioral.columns]
behavioral

# Plot distributions of fitness stats

For each column of the behavioral dataframe, divide participants based on behavioral performance (using `bounds`).  Then plot the distribution of each fitness variable.  Create one panel for each fitness variable, where each column has one distribution per group of particiapants (divided by behavior for that task).

Full figure:
  - Rows: behavioral columns
  - Columns: fitness vars
  - Each panel: distributions of fitness var-- one for each group (as defined by bounds applied to behavioral var)
  
Create 6 figures-- each combination of:
  - Immediate vs. delayed (behavioral)
  - Static vs. recent vs. baseline (fitness)

In [ ]:
def split_distplots(vals, group_vals, bounds, cmap, bins=10, xlabel='Value', ylabel='Number of participants', fontsize=14, **kwargs):
    x = pd.DataFrame(vals)            
    x['group'] = np.digitize(group_vals, get_bounds(group_vals, bounds=bounds))
    x = x.query(f'group < {len(bounds)}').dropna(axis=0, how='any')        
    
    sns.histplot(x, x=x.columns[0], hue='group', palette=cmap[:len(np.unique(x['group']))],
                 fill=True, legend=False, bins=bins, **kwargs)
    
    ax = kwargs.pop('ax', plt.gca())
    if len(xlabel) > 0:
        ax.set_xlabel(xlabel, fontsize=fontsize)
    if len(ylabel) > 0:
        ax.set_ylabel(ylabel, fontsize=fontsize)

In [ ]:
def split_scatterplots(vals, group_vals, bounds, cmap, xlabel='', ylabel='', fontsize=14, **kwargs):    
    sns.scatterplot(x=vals.values.ravel(), y=group_vals.values.ravel(), hue=group_vals.values.ravel(), palette=cmap,
                    legend=False, **kwargs)
    
    ax = kwargs.pop('ax', plt.gca())
    if len(xlabel) > 0:
        ax.set_xlabel(xlabel, fontsize=fontsize)
    if len(ylabel) > 0:
        ax.set_ylabel(ylabel, fontsize=fontsize)

In [ ]:
def grid_helper(behavioral, fitbit, bounds, task_cmaps, plotfun, **kwargs):
    fig, axes = plt.subplots(nrows=len(behavioral.columns), ncols=len(fitbit.columns),
                             sharex='col', sharey='row', figsize=(2*len(fitbit.columns), 2*len(behavioral.columns)))

    for i, b in enumerate(behavioral.columns):                    
        cmap = [v for k, v in task_cmaps.items() if k in b][0]  # get this task's colormap        
        for j, f in enumerate(fitbit.columns):
            if i == len(behavioral.columns) - 1:
                xlabel = format_text(f)
            else:
                xlabel = ''    
            
            if j == 0:
                ylabel = format_text(b)
            else:
                ylabel = ''
                        
            plotfun(pd.DataFrame(index=fitbit.index, columns=[format_text(f)], data=fitbit[f].values),
                    behavioral[b], bounds, cmap=cmap, ax=axes[i, j], 
                    xlabel=xlabel, ylabel=ylabel, fontsize=8,
                    **kwargs)

    return fig

In [ ]:
def distplot_grid(behavioral, fitbit, bounds, task_cmaps, bins=10, **kwargs):
    return grid_helper(behavioral, fitbit, bounds, task_cmaps, split_distplots, bins=bins, **kwargs)

In [ ]:
def scatterplot_grid(behavioral, fitbit, bounds, task_cmaps, **kwargs):
    return grid_helper(behavioral, fitbit, bounds, task_cmaps, split_scatterplots, **kwargs)

In [ ]:
def grid_plotter(behavioral, fitbit, behavioral_tags, fitness_tags, gridplotter, task_cmaps, **kwargs):
    for task_cond in np.unique(behavioral_tags):
        b = behavioral.loc[:, np.array(behavioral_tags) == task_cond]
        for fit_cond in np.unique(fitness_tags):
            f = fitbit.loc[:, np.array(fitness_tags) == fit_cond]

            plt.clf()
            fig = gridplotter(b, f, bounds, task_cmaps, **kwargs)

            fname = f'{task_cond}_{fit_cond}_{gridplotter.__name__}.pdf'
            fig.savefig(fname)

In [ ]:
grid_plotter(behavioral, fitbit, behavioral_tags, fitness_tags, distplot_grid, task_cmaps, bins=15)

In [ ]:
grid_plotter(behavioral, fitbit, behavioral_tags, fitness_tags, scatterplot_grid, task_cmaps_extended)

# Plot distributions of fitness variables

In [ ]:
color_dict = {f: c for f, c in zip(np.unique(fitness_tags), ['#A81A8A', '#1D2A7C', '#B29802'])}


for i, c in enumerate(fitbit.columns):
    plt.clf()
    fig = plt.figure(figsize=(3, 3))
    
    sns.displot(fitbit[c], kde=True, color=color_dict[fitness_tags[i]])
    plt.ylabel('Number of participants', fontsize=14)
    plt.xlabel(format_text(c, width=50), fontsize=14)
    
    plt.savefig(f"{c.replace(':', '-').replace('/', '-').replace(' ', '')}_dist.pdf")
    plt.close()

In [ ]:
fitbit

In [ ]:
with open(os.path.join(DATA_DIR, 'fitness_summary.pkl'), 'wb') as f:
    pickle.dump(fitbit, f)